# 14. Preparing the data for the LSTM (22-day windows)
**Part 3 — The LSTM.** The class models (decision tree, random forest, SVR) look at **one row** at a time. An LSTM looks at a **sequence**: for each forecast it reads the last **22 trading days** (about a month) of the 6 inputs — a small table of 22 rows × 6 columns. This notebook builds those tables for the 2007 model.

* **Needs:** `work/model_data.csv`. **Makes:** nothing new (notebook 15 repeats these steps and trains the model).

*Simple notebook series of the project 'This Time Is Different?'. Prepared with AI assistance (declared in `notes/ai_use.md`); a study notebook, not dissertation text.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder = '/content/drive/MyDrive/DataSets/ThisTimeIsDifferent/'

import warnings
warnings.filterwarnings('ignore')     # hide warning messages

In [2]:
from pandas import read_csv
dataset = read_csv(folder + 'work/model_data.csv', index_col=0, parse_dates=True)
x = dataset[['ret', 'log_r2', 'us_ret', 'log_us_r2', 'euro_ret', 'log_euro_r2']]
y = dataset['log_rv5_fwd']
print(x.shape)
print(y.shape)

(5243, 6)
(5243,)


## What is a 'window'? A small example
Ten days of one input, and windows of 3 days. The window for day 5 is days 3, 4 and 5 — always **up to and including** the forecast day, never after it.

In [3]:
from numpy import arange
small = arange(1, 11)
print('all days         :', small)
print('window for day 5 :', small[2:5])
print('window for day 10:', small[7:10])

all days         : [ 1  2  3  4  5  6  7  8  9 10]
window for day 5 : [3 4 5]
window for day 10: [ 8  9 10]


## Three groups of days for the 2007 model
* **fitting days** — the model learns from them (2003 – 2005);
* **validation days** — the last 12 months before the test year (2006). They are **not** learned from; they tell us **when to stop training** and give the smearing factor;
* **test days** — 2007, the forecasts we score.

Rules: the first 21 days cannot be used (their 22-day window would start before the data), and the last 5 days of each group are dropped (their target reaches into the next group).

In [4]:
fit_days = y.iloc[21:].loc[:'2005-12-31'].dropna().index
fit_days = fit_days[:-5]
val_days = y.loc['2006-01-01':'2006-12-31'].dropna().index
val_days = val_days[:-5]
test_days = y.loc['2007-01-01':'2007-12-31'].dropna().index
print('fitting   :', len(fit_days), 'days,', fit_days[0].date(), 'to', fit_days[-1].date())
print('validation:', len(val_days), 'days,', val_days[0].date(), 'to', val_days[-1].date())
print('test      :', len(test_days), 'days,', test_days[0].date(), 'to', test_days[-1].date())

fitting   : 734 days, 2003-01-31 to 2005-12-21
validation: 248 days, 2006-01-03 to 2006-12-20
test      : 254 days, 2007-01-02 to 2007-12-31


## Scaling (fitted on the fitting days only)
The same idea as `StandardScaler` in notebook 07: subtract the mean and divide by the standard deviation, both learned **only** from the data up to the last fitting day. The target y is scaled too (and the forecast is turned back at the end).

In [5]:
mean = x.loc[:fit_days[-1]].mean()
std = x.loc[:fit_days[-1]].std()
x_scaled = (x - mean) / std

y_mean = y.loc[fit_days].mean()
y_std = y.loc[fit_days].std()
y_scaled = (y - y_mean) / y_std

In [6]:
x_scaled.loc['2003-01-01':'2005-12-31'].describe().round(2)

,ret,log_r2,us_ret,log_us_r2,euro_ret,log_euro_r2
count,760.00,760.00,760.00,760.00,760.00,760.00
mean,0.00,0.00,-0.00,-0.00,-0.00,-0.01
std,1.00,1.00,1.00,1.00,1.00,1.01
min,-7.67,-3.22,-4.37,-3.28,-4.75,-3.30
25%,-0.55,-0.47,-0.63,-0.55,-0.44,-0.57
50%,0.03,0.19,0.03,0.20,0.01,0.13
75%,0.57,0.71,0.56,0.73,0.50,0.73
max,4.33,2.48,4.13,1.95,5.19,2.12


## One window
The window of one fitting day: `get_loc` gives the row number of the day, and we take the 22 rows ending at it.

In [7]:
day = fit_days[100]
position = x.index.get_loc(day)
window = x_scaled.iloc[position - 21:position + 1]
print('forecast day:', day.date())
window

forecast day: 2003-06-26


,ret,log_r2,us_ret,log_us_r2,euro_ret,log_euro_r2
2003-05-27,-0.720493,0.260358,2.275417,1.407096,1.121631,0.886938
2003-05-28,-0.300228,-0.736050,0.163556,-0.694081,1.118934,0.885044
2003-05-29,-0.787152,0.350635,-0.509058,-0.051260,-0.381814,-0.173185
2003-05-30,-0.075931,-2.747651,1.698465,1.154154,1.855057,1.286907
2003-06-03,3.083648,1.708594,0.509801,0.147507,-0.973325,0.681186
2003-06-04,0.455122,0.152495,1.748253,1.179048,1.234690,0.962674
2003-06-05,0.525453,0.259039,0.418605,-0.008796,-1.030059,0.730732
2003-06-06,0.045294,-1.052201,-0.345877,-0.448932,2.050733,1.367435
2003-06-09,-0.968684,0.558176,-1.506131,0.984311,-0.834050,0.545236
2003-06-10,-1.948863,1.227860,1.037306,0.733408,1.027124,0.817824


## All windows
A `for` loop does this for every fitting day and puts the windows in a list; `array` turns the list into one 3-dimensional block: (number of days, 22 days, 6 inputs).

In [8]:
from numpy import array
x_fit = []
for day in fit_days:
    position = x.index.get_loc(day)
    x_fit.append(x_scaled.values[position - 21:position + 1])
x_fit = array(x_fit, dtype='float32')
y_fit = array(y_scaled.loc[fit_days], dtype='float32')
print(x_fit.shape)
print(y_fit.shape)

(734, 22, 6)
(734,)


So the 2007 model learns from 734 examples; each example is a 22 × 6 table, and its answer is one number (the scaled log volatility of the following week). Notebook 15 builds and trains the LSTM on exactly these.